# Trabalho 1 de IA: Busca em Espaço de Estados
## Problema: Modelagem e Busca em Redes Elétricas (IEEE 14-Bus)

Este notebook contém o relatório completo e a implementação do Trabalho 1 de IA, focado na modelagem de redes elétricas como grafos e na aplicação de algoritmos de busca.

### 1. Modelagem do Problema e Visualizador do Grafo

Abaixo definimos os componentes principais do ambiente e a função para **renderização 2D do grafo**. Isso permite uma visualização visual das soluções geradas pelas buscas.

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Inicializa o grafo (Exemplo temporário para testar a interface)
G = nx.fast_gnp_random_graph(14, 0.3, seed=42) # A ser substituido pelos dados reais IEEE 14
node_names = [f"Bus {i}" for i in range(14)]
mapping = {i: node_names[i] for i in range(14)}
G = nx.relabel_nodes(G, mapping)

# Posicionamento padrão para visualização
pos = nx.spring_layout(G, seed=42)

def render_graph_2d(graph, path=None, title="Visão Geral da Rede"):
    """
    Plota o grafo. Se um caminho (path) for fornecido, 
    destaca as arestas e nós pertencentes a esse caminho em vermelho.
    """
    plt.figure(figsize=(10, 6))
    
    # Desenha os nós e arestas básicos
    nx.draw_networkx_nodes(graph, pos, node_color="lightblue", node_size=600)
    nx.draw_networkx_labels(graph, pos, font_size=9, font_weight="bold")
    nx.draw_networkx_edges(graph, pos, edge_color="gray", alpha=0.6)
    
    # Destaca o caminho (Solução)
    if path:
        path_edges = list(zip(path, path[1:]))
        nx.draw_networkx_nodes(graph, pos, nodelist=path, node_color="salmon", node_size=700)
        nx.draw_networkx_edges(graph, pos, edgelist=path_edges, edge_color="red", width=2.5)
        
    plt.title(title)
    plt.axis("off")
    plt.show()

### 2. Algoritmos de Busca

Implementação das funções de busca que serão comparadas, separadas em categorias.

In [ ]:
###################################################################
# BLOC 1: BUSCAS CEGAS (NÃO INFORMADAS)
###################################################################

def bfs_search(graph, start, goal):
    """Busca em Largura (Breadth-First Search)"""
    try:
        # Usa o nx nativo temporariamente para retornar o caminho heurístico base
        return nx.shortest_path(graph, source=start, target=goal)
    except nx.NetworkXNoPath:
        return []

def dfs_search(graph, start, goal):
    """Busca em Profundidade (Depth-First Search)"""
    for path in nx.all_simple_paths(graph, source=start, target=goal):
        return path # Retorna o primeiro caminho encontrado
    return []

###################################################################
# BLOC 2: BUSCA INFORMADA (HEURÍSTICA)
###################################################################

def custom_heuristic(node_a, node_b):
    """Heurística customizada (ex: distância baseada no layout para teste)"""
    (x1, y1) = pos[node_a]
    (x2, y2) = pos[node_b]
    return ((x1 - x2)**2 + (y1 - y2)**2)**0.5

def astar_search(graph, start, goal):
    """A* Search"""
    try:
        return nx.astar_path(graph, source=start, target=goal, heuristic=custom_heuristic)
    except nx.NetworkXNoPath:
        return []

### 3. Interface Interativa de Visualização

Utlize os **menus suspensos** abaixo para definir o **Início** e o **Destino**. Defina o algoritmo e clique em rodar para visualizar o caminho.

In [ ]:
out = widgets.Output()

dropdown_start = widgets.Dropdown(options=node_names, description='Início:', value=node_names[0])
dropdown_goal = widgets.Dropdown(options=node_names, description='Destino:', value=node_names[-1])
dropdown_algo = widgets.Dropdown(
    options=['BFS (Cega)', 'DFS (Cega)', 'A* (Informada)'], 
    description='Algoritmo:', 
    value='BFS (Cega)'
)
btn_run = widgets.Button(description='Executar Busca', button_style='success')

def on_button_clicked(b):
    with out:
        clear_output(wait=True)
        start_node = dropdown_start.value
        goal_node = dropdown_goal.value
        algo = dropdown_algo.value
        
        print(f"-> Iniciando busca de {start_node} para {goal_node} usando {algo}...")
        
        path = []
        if algo == 'BFS (Cega)':
            path = bfs_search(G, start_node, goal_node)
        elif algo == 'DFS (Cega)':
            path = dfs_search(G, start_node, goal_node)
        elif algo == 'A* (Informada)':
            path = astar_search(G, start_node, goal_node)
            
        if path:
            print(f"✅ Caminho encontrado: {' -> '.join(path)}")
            render_graph_2d(G, path=path, title=f"Resultado: {algo}")
        else:
            print("❌ Nenhum caminho encontrado.")

btn_run.on_click(on_button_clicked)

ui = widgets.VBox([widgets.HBox([dropdown_start, dropdown_goal]), dropdown_algo, btn_run])
display(ui, out)